# 07 - Joint-training oracle on the capped 12-label splits (diagnostic)
Pre-registration: `experiments/prereg_oracle.md` (committed before any oracle run). Code: `s11_oracle.py`.

Question: the 12-label continual runs learned only 6-8 of 12 labels. Is that continual forgetting / interference,
or can these features not separate DNS/LDAP and SSDP/UDP even when all classes are trained **together**?
One model is trained on all 159,595 training rows of the all12 runs at once (no updates, no gate, no buffer), with
the same scaler, architecture and loss, and is scored on the **same protected validation rows**. The verdict is
decided on validation by the registered rule. Test is scored once, later, only after the validation files are
committed.

Cells: CII e10 (the E0 recipe) and e20 x seeds 42-44, plus CI e10/e20 seed 42. 8 trainings, about 20-25 min on a T4.

**Inputs (Add Input):** notebook 04's output (as for notebooks 05 and 06): `nids_out/` with `clean_*.parquet`,
`all12/` and `runs/{cii,ci}/all12_*_s42`. **Internet ON**. Accelerator **GPU T4**.
Run interactively. Cells 7-8 do nothing unless `RESTORE` / `DECISIONS_COMMIT` are set.

In [ ]:
# 1. Locate code and the notebook-04 outputs (read only)
import glob, json, os, subprocess, sys, time, zipfile
from pathlib import Path
import pandas as pd
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)

CLONE = Path("/kaggle/working/code")
if not CLONE.exists() and os.environ.get("NIDS_CLONE", "1") == "1":
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/DevChi1l/cicddos2019-continual-nids",
                    str(CLONE)], check=False)
CODE_DIR = os.environ.get("NIDS_CODE") or next(
    (str(Path(p).parent) for p in [str(CLONE / "s11_oracle.py")] +
     sorted(glob.glob("/kaggle/input/**/s11_oracle.py", recursive=True)) if Path(p).exists()), None)
assert CODE_DIR, "code with s11_oracle.py not found: turn Internet ON"
sys.path.insert(0, CODE_DIR)
import s5_explain as X
git = subprocess.run(["git", "-C", CODE_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("code:", CODE_DIR, "| git commit:", git)

SRC = X.find_dir("runs/cii/all12_cii_s42/run_manifest.json", "NIDS_ORACLE_SRC")
assert SRC and (SRC / "clean_train.parquet").exists() and (SRC / "runs/ci/all12_ci_s42/run_manifest.json").exists(), \
    "attach notebook 04's output: nids_out/ with clean_*.parquet, all12/ and both all12 runs"
WORK = Path(os.environ.get("NIDS_WORK", "/kaggle/working/nids_out"))
OR = WORK / "oracle"
print("source (read only):", SRC, "\nwrites:", OR)

def run(*args):
    cmd = [sys.executable, str(Path(CODE_DIR) / "s11_oracle.py"), *map(str, args)]
    print("$", " ".join(cmd[1:]), flush=True)
    subprocess.run(cmd, check=True, cwd=CODE_DIR)

COMMON = ["--src", SRC, "--work", WORK]

def zip_oracle(name, test=False):
    # JSON/CSV under oracle/ (no models) -> /kaggle/working/<name>; needs only this cell
    dest = WORK.parent / name
    pick = sorted(p for p in OR.rglob("*") if p.is_file() and p.suffix in (".json", ".csv")
                  and (test or p.name != "test.json"))
    with zipfile.ZipFile(dest, "w", zipfile.ZIP_DEFLATED) as zf:
        for p in pick:
            zf.write(p, Path("oracle") / p.relative_to(OR))
    print(f"{len(pick)} files -> {dest} ({dest.stat().st_size / 1e6:.2f} MB). Download it from the Output panel.")

In [ ]:
# 2. GPU check
import torch
assert torch.cuda.is_available(), "Settings -> Accelerator -> GPU T4"
print(torch.cuda.get_device_name(0), torch.__version__)

In [ ]:
# 3. Train the 8 oracle cells (validation only; finished cells are skipped if re-run)
t0 = time.time()
run("train", *COMMON)
print(f"train: {time.time() - t0:.0f} s")

In [ ]:
# 4. The pre-registered verdict rule -> oracle/oracle_val.csv, oracle/verdict.json (validation only)
run("decide", "--work", WORK)

## STOP - download BOTH zips from the Output panel (cell 6 writes them)
- `oracle_val.zip`: validation JSON/CSV only (no models, no test). Send it; it is committed under
  `experiments/results/07_oracle/`.
- `oracle_restore.zip`: models + validation files + sha256 manifest (no test). Keep it for a new session.

Test is scored once, only after the commit: set `DECISIONS_COMMIT` in cell 8 and run cells 8-9. If the session
closed: new session, run cell 1, then the restore cell (7), then cells 8-9. Never cells 2-4 there.

In [ ]:
# 6. Validation outputs: oracle_val.zip (JSON/CSV only) and oracle_restore.zip (models + manifest); no test
assert not list(OR.rglob("test.json")), "test was already scored in this folder"
zip_oracle("oracle_val.zip")
run("bundle", "--work", WORK, "--bundle", WORK.parent / "oracle_restore.zip")

In [ ]:
# 7. RESTORE (only in a NEW session after the old one closed; skip otherwise)
RESTORE = False
if RESTORE:
    man = [m for m in sorted(glob.glob("/kaggle/input/**/RESTORE_MANIFEST.json", recursive=True))
           if (Path(m).parent / "oracle").is_dir()] or sorted(glob.glob("/kaggle/input/**/oracle_restore.zip", recursive=True))
    assert man, "attach the dataset made from oracle_restore.zip"
    b = Path(man[0])
    run("restore", "--work", WORK, "--bundle", b.parent if b.name == "RESTORE_MANIFEST.json" else b)
else:
    print("restore skipped (RESTORE = False)")

In [ ]:
# 8. Test, ONCE. Disabled until DECISIONS_COMMIT is the sha of the commit that holds oracle_val.zip's files.
DECISIONS_COMMIT = ""
if DECISIONS_COMMIT:
    subprocess.run(["git", "-C", CODE_DIR, "fetch", "--depth", "50", "origin", "main"], check=True)
    run("score-test", *COMMON, "--decisions-commit", DECISIONS_COMMIT, "--repo", CODE_DIR)
else:
    print("skipped: commit the validation files first, then set DECISIONS_COMMIT")

In [ ]:
# 9. Test files (after cell 8 ran; original or restore session - needs only cell 1)
done = sorted(OR.rglob("test.json"))
if not done:
    print("no test scored yet")
else:
    assert len(done) == 8, f"test scored for {len(done)} of 8 cells: run cell 8"
    zip_oracle("oracle_test.zip", test=True)